# Phase 6: Econometric Analysis & Model Assumption Audit

## Project: Credit Risk Modelling & Independent Model Validation
**Target Role**: Quantitative Risk Analytics / Credit Risk Model Validation

### Scope of Notebook
- Part 5: Heteroskedasticity Tests & Robust Covariance (Breusch-Pagan, White, Goldfeld-Quandt, HC3 SEs)
- Part 6: Autocorrelation Analysis (Durbin-Watson, Breusch-Godfrey)
- Part 7: Linearity Diagnostics (Box-Tidwell, LOWESS, Partial Residuals)
- Part 8: Outlier & Influence Diagnostics (Hat values, Cook's D, Studentized Residuals)
- Part 9: Endogeneity & IV/2SLS Evaluation Framework

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from validation.econometrics import (
    run_heteroskedasticity_tests,
    run_autocorrelation_tests,
    evaluate_endogeneity_framework,
)
from validation.diagnostics import (
    run_box_tidwell_test,
    calculate_outlier_influence_metrics,
)

pd.set_option("display.max_columns", 30)
print("Econometric validation modules loaded successfully!")

Econometric validation modules loaded successfully!


In [2]:
data_path = Path.cwd().parent / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
cols_to_use = [
    "loan_status", "issue_d", "loan_amnt", "int_rate", "installment", "annual_inc",
    "dti", "fico_range_low", "revol_util", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "total_acc",
    "fe_loan_to_income_ratio", "fe_monthly_installment_to_income_ratio",
    "fe_credit_utilization", "fe_available_revolving_credit", "fe_credit_exposure",
    "fe_debt_burden", "fe_credit_history_months"
]
df = pd.read_csv(data_path, usecols=cols_to_use, nrows=100000, low_memory=False)

bad_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
good_statuses = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
df["target"] = np.nan
df.loc[df["loan_status"].isin(bad_statuses), "target"] = 1.0
df.loc[df["loan_status"].isin(good_statuses), "target"] = 0.0

df_model = df.dropna(subset=["target"]).copy()
print(f"Binary dataset size for Econometrics: {len(df_model):,}")

Binary dataset size for Econometrics: 88,333


In [3]:
numeric_features = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low", "revol_util", "delinq_2yrs"]
het_summary, hc3_table = run_heteroskedasticity_tests(df_model, "target", numeric_features)
print("Heteroskedasticity Summary:", het_summary)
print("HC3 Robust SE Table:")
print(hc3_table)

Heteroskedasticity Summary: {'breusch_pagan_lm_stat': 1681.5638, 'breusch_pagan_pvalue': 0.0, 'white_lm_stat': nan, 'white_pvalue': nan, 'goldfeld_quandt_stat': 1.0604, 'goldfeld_quandt_pvalue': 0.005611630284034258, 'heteroskedasticity_present': 'Yes'}
HC3 Robust SE Table:
          feature      coef  ols_std_err  hc3_robust_std_err  \
0           const  0.414117     0.091809            0.084947   
1       loan_amnt  0.000017     0.000001            0.000001   
2        int_rate  0.025614     0.000843            0.001013   
3     installment -0.000492     0.000042            0.000048   
4      annual_inc -0.000000     0.000000            0.000000   
5             dti  0.002279     0.000308            0.001162   
6  fico_range_low -0.000811     0.000123            0.000114   
7      revol_util -0.000477     0.000143            0.000158   
8     delinq_2yrs -0.001991     0.003471            0.003660   

   se_difference_pct     ols_pvalue     hc3_pvalue  
0              -7.47   6.512657

In [4]:
ac_summary = run_autocorrelation_tests(df_model, "target", numeric_features, time_col="issue_d")
ac_summary

{'durbin_watson_stat': 1.9549,
 'durbin_watson_interpretation': 'No substantial first-order autocorrelation (DW ~ 2.0)',
 'breusch_godfrey_lm_stat': 13.3191,
 'breusch_godfrey_pvalue': 0.009817535390071982,
 'autocorrelation_present': 'Yes'}

In [5]:
box_tidwell_table = run_box_tidwell_test(df_model, "target", numeric_features)
box_tidwell_table

,feature,interaction_coef,box_tidwell_pvalue,is_non_linear
0,loan_amnt,-0.000032,6.414469e-04,Yes
1,int_rate,-0.260503,7.064822e-15,Yes
2,installment,-0.001890,2.144642e-09,Yes
3,annual_inc,0.000002,4.793385e-02,Yes
4,fico_range_low,0.018213,5.939039e-01,No


In [6]:
_, influence_summary = calculate_outlier_influence_metrics(df_model, "target", numeric_features)
pd.DataFrame([influence_summary])

,sample_n,parameters_p,leverage_threshold_2p_n,high_leverage_observations,high_leverage_pct,cooks_distance_threshold_4_n,influential_cooks_observations,influential_cooks_pct,outlier_studentized_resids_abs3,outlier_studentized_pct,banking_recommendation
0,5000,9,0.0036,304,6.08,0.0008,273,5.46,0,0.0,Do NOT remove observations blindly. Leverage c...


In [7]:
endo_eval = evaluate_endogeneity_framework()
for k, v in endo_eval.items():
    print(f"[{k.upper()}]: {v}")

[ENDOGENEITY_SOURCES]: 1. Simultaneous determination: Interest rate is set based on borrower risk, but interest rate also drives borrower default risk.
2. Omitted variable bias: Unobserved borrower attributes (wealth, financial discipline, job security) influence both income reporting and repayment capability.
3. Measurement error: Self-reported annual income and DTI in bureau data contain measurement noise.
[IV_2SLS_SUITABILITY_ASSESSMENT]: Instrumental Variables (IV) and Two-Stage Least Squares (2SLS) are NOT appropriate for this project due to the absence of valid exogenous instruments in observational credit bureau data.
A valid instrument Z must satisfy two strict conditions:
  a. Instrument Relevance: Cov(Z, X) != 0
  b. Instrument Exogeneity / Exclusion Restriction: Cov(Z, u) = 0 (Z affects Default ONLY through X, with no direct channel).
In retail lending datasets like LendingClub, potential candidate instruments (e.g. macro interest rates, zip-code economic proxies) fail the e